# Polygon conservative regridder — curvilinear target

Curvilinear grids have 2D `lat(y,x)` / `lon(y,x)` coordinate arrays — common in
ocean models (ORCA, tripolar) and regional forecasts on rotated grids. The existing
`.conservative` method is strictly rectilinear, so `ConservativeRegridder` is the
available tool here.

This notebook regrids a regular lat/lon source onto a 30°-rotated curvilinear target.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder

## Source — regular 1° lat/lon

A simple two-bump analytic field makes the geometry easy to read off the plot.

In [ ]:
lat = np.linspace(-60, 60, 121)
lon = np.linspace(-120, 120, 241)
Lo, La = np.meshgrid(lon, lat)
field = (
    np.exp(-((Lo - 40)**2 + (La - 20)**2) / 500)
    - np.exp(-((Lo + 60)**2 + (La + 15)**2) / 400)
)
src = xr.DataArray(
    field,
    dims=("latitude", "longitude"),
    coords={"latitude": lat, "longitude": lon},
    name="bumps",
)
src.plot(figsize=(8, 3.5), cmap="RdBu_r", center=0)
plt.title("source: analytic two-bump field")
plt.tight_layout()

## Target — rotated curvilinear grid

Coordinates ride on a `(ny, nx)` mesh rather than a 1D lat/lon. We stash them
as 2D coordinate variables on an `xr.Dataset`.

In [ ]:
ny_t, nx_t = 30, 50
xi, yi = np.meshgrid(
    np.linspace(-110, 110, nx_t),
    np.linspace(-45, 45, ny_t),
    indexing="xy",
)
theta = np.deg2rad(30)
lon2d = xi * np.cos(theta) - yi * np.sin(theta)
lat2d = xi * np.sin(theta) + yi * np.cos(theta)

target = xr.Dataset(
    coords={
        "longitude": (("ny", "nx"), lon2d),
        "latitude":  (("ny", "nx"), lat2d),
    }
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lon2d, lat2d, color="0.3", lw=0.4)
ax.plot(lon2d.T, lat2d.T, color="0.3", lw=0.4)
ax.set_title("curvilinear target (30° rotation)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal")

## Regrid

The accessor detects that `latitude` / `longitude` are 2D and routes through
the curvilinear path (threaded GEOS polygon clipping under the hood). On the
reusable class you'd construct once via `ConservativeRegridder(src, target, ...)`
and apply to many fields.

In [ ]:
regridded = src.regrid.conservative_polygon(
    target, x_coord="longitude", y_coord="latitude"
)
regridded

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
pc = ax.pcolormesh(lon2d, lat2d, regridded.values, cmap="RdBu_r", shading="auto",
                   vmin=-1, vmax=1)
fig.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title("regridded onto rotated curvilinear grid")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_aspect("equal")

## Conservation check

The regridder's internal intersection-area matrix `A[i, j]` gives the exact
mass that each source cell contributes to each target cell. Summing those
contributions against the source field is the ground-truth mass within the
overlap region. Multiplying the output by each target cell's covered area
should match.

Target cells that fall outside the source are NaN in the output, which is
why we mask them before summing.

In [ ]:
rgr = ConservativeRegridder(src, target, x_coord="longitude", y_coord="latitude")
A = rgr._areas  # sparse (n_dst, n_src)
target_covered = A.sum(axis=1).todense().reshape(regridded.shape)
src_covered = A.sum(axis=0).todense()

valid = np.isfinite(regridded.values)
direct_mass = float((src.values.ravel() * src_covered).sum())
regridded_mass = float((regridded.values[valid] * target_covered[valid]).sum())
print(f"direct     (A · s):          {direct_mass:.6f}")
print(f"regridded  (out · a_dst):    {regridded_mass:.6f}")
print(f"relative difference:         {abs(direct_mass - regridded_mass) / max(abs(direct_mass), 1e-12):.2e}")
print(f"covered target fraction:     {valid.mean():.2%}")